# SDR Firmware Architecture & Technical Approach
## CPTR 480 Final Presentation - Joseph Pennock

The firmware for this Software Defined Radio (SDR) receiver is built upon the Raspberry Pi Pico SDK. It utilizes a dual-core, non-blocking architecture to ensure uninterrupted high-fidelity audio streaming alongside responsive standalone user controls and USB serial communications.

---

## Newhaven 1x8 Alphanumeric LCD Driver

This software driver provides a hardware abstraction layer (HAL) for controlling a standard HD44780-compliant 1x8 alphanumeric LCD display using the Raspberry Pi Pico SDK. The driver operates exclusively in 8-bit parallel mode to minimize command overhead and maximize refresh execution speeds.

## API Reference

### `void lcd_init(void)`
Initializes the configured GPIO pins and executes the standard HD44780 3-stage software reset/wakeup sequence.
* **Execution:** Blocking.
* **Internal Delays:** Employs a baseline 100 ms startup delay to allow the LCD's internal controller to stabilize, followed by stepped intervals (30 ms, 10 ms) during initialization.
* **Default Display Configurations:**
    * Function Set: 8-bit mode, 1-line display configuration (`0x38`)
    * Display Control: Display ON, Cursor Blinking OFF (`0x0C`)
    * Screen State: Cleared automatically post-initialization.

### `void lcd_command(uint8_t cmd)`
Sends a raw 8-bit control instruction code directly to the display controller registers.
* **Parameters:** `cmd` - The 8-bit command opcode (e.g., screen shift, entry mode updates).
* **Timing:** Pulls the `RS` pin low and triggers the enable latch pulse.

### `void lcd_clear(void)`
Wipes the entire DDRAM address space of the display and returns the hardware cursor to the home position (top-left).
* **Execution:** Blocking.
* **Timing Safety:** Blocks execution for an explicit 2 ms window (`sleep_ms(2)`) to satisfy the processing time required by the HD44780 chip for an execution clear (`0x01`) instruction.

### `void lcd_print(const char* str)`
Writes a standard null-terminated C-string to the active line segment of the display interface.
* **Parameters:** `str` - Pointer to the character array.
* **Execution Mode:** Drives the `RS` pin high to switch the hardware bus to data mode before iteratively dispatching the character bytes.

## Timing Architecture

* **Latching Mechanism (`lcd_pulse_e`):** The driver enforces a 1 microsecond hold time on the active-high state of the Enable pin.
* **Bus Processing (`lcd_write_bus`):** Bitmask operations sequentially map the targeted byte matrix over the custom pin structure `DB_PINS` array in an LSB-to-MSB sequence alignment.

---

## Interrupt Service Routines (ISRs)

Real-time interrupt handlers responsible for background audio data capture, handoffs, and physical control interface tracking state.

## 1. DMA Ring Buffer Interrupt (`dma_handler`)

The `dma_handler` is an explicit, high-priority hardware Interrupt Service Routine (ISR) configured to run exclusively on **Core 1**. It coordinates the low-level execution boundaries of the dual-channel DMA ping-pong data stream coming from the I2S peripheral.

## 2. Rotary Quadrature Encoder Decoder (`encoder_callback`)

The `encoder_callback` is an asynchronous edge-triggered GPIO interrupt service routine executing on **Core 0**. It translates physical mechanical movements of the radio front-panel dial into tuning offsets.

### Technical Attributes
* **Trigger Mechanics:** Configured to monitor edge transitions exclusively on the encoder's clock/phase channel pin (`ENC_A_PIN`).
* **Hardware Interlock Matrix:**

| Trigger Edge (Pin A) | Read State (Pin B) | Decoded Direction | Frequency Mutation (`current_hz`) |
| :--- | :--- | :--- | :--- |
| **Rising Edge** | **Logic HIGH (1)** | Clockwise (CW) | **+ 40,000 Hz (+40 kHz)** |
| **Rising Edge** | **Logic LOW (0)** | Counter-Clockwise (CCW) | **- 40,000 Hz (-40 kHz)** |

### Operational State Logic
1. **Source Verification:** Filters incoming edge pulses to confirm the event generated from the true clock input pin (`ENC_A_PIN`).
2. **Directional Parsing:** Performs a direct instantaneous hardware register poll via `gpio_get()` on the trailing reference pin (`ENC_B_PIN`). If the line reads high, the user is turning the dial clockwise; a low reading indicates a counter-clockwise operation.
3. **VFO Registry Mutation:** Modifies the baseline frequency tracker `current_hz` by an offset of **40,000 Hz** per step.
4. **State Dirty Flag:** Asserts `frequency_changed = true`. This tells the background tracking thread in the main  loop to update the si5351 clock generator and output the modified readings to the LCD panel.

Note: The step scale increment parameter is explicitly hardcoded to 40,000 Hz, because the hardware design utilizes an intermediate 4:1 Johnson counter layout to generate native phase-quadrature ($0^\circ, 90^\circ, 180^\circ, 270^\circ$) intermediate signals for the Tayloe Mixer, this internal driver calculation maps actual radio tuning translation step of 10,000 Hz (10 kHz) to the receiver's alphanumeric display interface.

---

## UAC1 Device Enumeration

<div style="text-align: left;">
<img src="/Users/josephpennock/Documents/GitHub/ENGR-356-SDR/Images/UAC1Enum.png" width="450" height="270" />
<img src="/Users/josephpennock/Documents/GitHub/ENGR-356-SDR/Images/AudacityEnum.png" width="650" height="270" />
</div>

- **Audio Streaming:** This aspect of the project does not currently work. I am pretty sure there is some firmware issue between the DMA and the ADC, but just ran out of time debugging it. 

---

## (Mostly) Working SDR Firmware

<div style="text-align: left;">
<img src="/Users/josephpennock/Documents/GitHub/ENGR-356-SDR/Images/SDRinit.jpg" width="320" height="240" />
<img src="/Users/josephpennock/Documents/GitHub/ENGR-356-SDR/Images/SDRmin.JPG" width="320" height="240" />
<img src="/Users/josephpennock/Documents/GitHub/ENGR-356-SDR/Images/SDRmax.JPG" width="320" height="240" />
</div>
